# 02 — The Agent Card: How Agents Describe Themselves

## Why this notebook exists

In **notebook 01** we built two services that talked to each other through hand-rolled REST. We watched that fall over the moment one side changed its contract — because there was no machine-readable description of either service. Consumers had no way to ask: *"what skills do you have, what inputs do you take, and how do I authenticate?"*

A2A solves this with a single, simple primitive: the **Agent Card**. It's a JSON document served at a well-known URL (`/.well-known/agent-card.json`) that describes everything a client needs to know to start talking to an agent.

This notebook builds an Agent Card for the researcher from notebook 01, fetches it from a client, and walks through what the card actually says.

> *We target A2A spec v0.3.0 throughout. Earlier drafts of A2A used `/.well-known/agent.json` (no hyphen); the current spec uses `/.well-known/agent-card.json`.*

## What you'll learn

- The shape of an A2A **Agent Card**: `protocolVersion`, `name`, `description`, `url`, `capabilities`, `securitySchemes`/`security`, `defaultInputModes`/`defaultOutputModes`, and `skills`.
- How to serve `/.well-known/agent-card.json` from a FastAPI app.
- How to discover an agent's capabilities from the client side using `httpx`.
- How to parse a card into a typed `pydantic` model and iterate over its declared skills.
- Why the Agent Card is **not** the same as OpenAPI, and what each is good for.

## 1. Setup

Same pattern as notebook 01 — we run a FastAPI app on a background thread so we can both serve and call it from this notebook. Each notebook in this series is self-contained, so the helper is re-defined here rather than imported.

In [1]:
import json
import threading
import time

import httpx
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel, Field

_servers: list[uvicorn.Server] = []


def run_server_in_thread(app: FastAPI, port: int) -> uvicorn.Server:
    """Start `app` on localhost:`port` in a background daemon thread."""
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)

    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()

    for _ in range(50):
        if server.started:
            break
        time.sleep(0.05)
    else:
        raise RuntimeError(f"Server on port {port} did not start in time")

    _servers.append(server)
    return server


def shutdown_all_servers() -> None:
    for server in list(_servers):
        server.should_exit = True
    _servers.clear()


print("Setup OK")

Setup OK


## 2. What is an Agent Card?

An Agent Card is a JSON document an agent **publishes about itself**. The A2A spec mandates it lives at the path `/.well-known/agent-card.json` on the agent's base URL — the same `/.well-known/` convention used by OAuth, OpenID Connect, Web App Manifests, and other web standards.

A minimal card answers four questions:

| Question | Card field |
|---|---|
| *Who are you?* | `name`, `description`, `version`, `protocolVersion` |
| *Where do I send requests?* | `url` |
| *What can you do?* | `capabilities`, `skills[]`, `defaultInputModes`, `defaultOutputModes` |
| *How do I authenticate?* | `securitySchemes`, `security` (OpenAPI-style) |

The card is **machine-readable** — a client fetches it once at the start of an interaction and can then make sensible decisions: "this agent supports streaming, so I'll use `message/stream` instead of `message/send`," or "this agent requires a bearer token, so I'll attach one."

The card is **also human-readable** — `description` and `skills[].description` are free-form prose meant to help humans (and LLMs reading the card on their behalf) understand what the agent is for.

## 3. Serve the Researcher's Agent Card

Now we build the server side. We'll:

1. Define `pydantic` models matching the A2A v0.3.0 card schema.
2. Build a FastAPI app with one endpoint: `GET /.well-known/agent-card.json`.
3. Start it on `127.0.0.1:8010`.

The card declares one skill — `research_topic` — corresponding to the researcher's only ability. We leave `securitySchemes` and `security` empty for now (no auth required); notebook 07 will fill them in.

In [2]:
class AgentCapabilities(BaseModel):
    streaming: bool = False
    pushNotifications: bool = False
    stateTransitionHistory: bool = False


class AgentSkill(BaseModel):
    id: str
    name: str
    description: str
    tags: list[str] = Field(default_factory=list)
    examples: list[str] = Field(default_factory=list)


class AgentCard(BaseModel):
    # A2A spec v0.3.0. protocolVersion is optional with a default; defaultInputModes
    # and defaultOutputModes are required; securitySchemes/security are optional
    # (we leave them empty until notebook 07).
    protocolVersion: str = "0.3.0"
    name: str
    description: str
    url: str
    version: str
    capabilities: AgentCapabilities = Field(default_factory=AgentCapabilities)
    defaultInputModes: list[str] = Field(default_factory=lambda: ["text"])
    defaultOutputModes: list[str] = Field(default_factory=lambda: ["text"])
    skills: list[AgentSkill] = Field(default_factory=list)
    # OpenAPI-style auth modeling. Both default to "no auth required."
    securitySchemes: dict[str, dict] = Field(default_factory=dict)
    security: list[dict[str, list[str]]] = Field(default_factory=list)


researcher_app = FastAPI()


RESEARCHER_CARD = AgentCard(
    name="Researcher",
    description="Returns canned facts on a small set of well-known topics.",
    url="http://127.0.0.1:8010",
    version="0.1.0",
    skills=[
        AgentSkill(
            id="research_topic",
            name="Research a topic",
            description="Given a topic name, return a list of facts about it.",
            tags=["research", "facts"],
            examples=["octopuses", "rome"],
        ),
    ],
)


@researcher_app.get("/.well-known/agent-card.json", response_model=AgentCard)
def agent_card() -> AgentCard:
    return RESEARCHER_CARD


researcher_server = run_server_in_thread(researcher_app, port=8010)
print("Researcher running on http://127.0.0.1:8010")

Researcher running on http://127.0.0.1:8010


In [3]:
resp = httpx.get("http://127.0.0.1:8010/.well-known/agent-card.json")
print(resp.status_code)
print(json.dumps(resp.json(), indent=2))

200
{
  "protocolVersion": "0.3.0",
  "name": "Researcher",
  "description": "Returns canned facts on a small set of well-known topics.",
  "url": "http://127.0.0.1:8010",
  "version": "0.1.0",
  "capabilities": {
    "streaming": false,
    "pushNotifications": false,
    "stateTransitionHistory": false
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "skills": [
    {
      "id": "research_topic",
      "name": "Research a topic",
      "description": "Given a topic name, return a list of facts about it.",
      "tags": [
        "research",
        "facts"
      ],
      "examples": [
        "octopuses",
        "rome"
      ]
    }
  ],
  "securitySchemes": {},
  "security": []
}


## 4. Discover the Agent from a Client

A client that wants to talk to an agent does one thing first: **fetch the Agent Card**. With nothing more than the agent's base URL, it can learn the agent's identity, capabilities, and skills.

We'll parse the card back into the same `AgentCard` pydantic model we defined on the server side. In a real cross-team setup the client wouldn't have access to the server's classes — it would have its own definitions of the same A2A schema, or use a shared SDK. The shapes are governed by the A2A spec, not by either side's code.

In [4]:
def discover_agent(base_url: str) -> AgentCard:
    """Fetch and parse an agent's card from its base URL."""
    resp = httpx.get(f"{base_url.rstrip('/')}/.well-known/agent-card.json")
    resp.raise_for_status()
    return AgentCard.model_validate(resp.json())


card = discover_agent("http://127.0.0.1:8010")
print(f"Found agent: {card.name} (v{card.version}, protocol v{card.protocolVersion})")
print(f"  Description: {card.description}")
print(f"  Endpoint:    {card.url}")
print(f"  Streaming?   {card.capabilities.streaming}")
print(f"  # skills:    {len(card.skills)}")
print(f"  Auth:        {'none' if not card.security else card.security}")

Found agent: Researcher (v0.1.0, protocol v0.3.0)
  Description: Returns canned facts on a small set of well-known topics.
  Endpoint:    http://127.0.0.1:8010
  Streaming?   False
  # skills:    1
  Auth:        none


## 5. Inspect the Agent's Skills

The card's `skills[]` field is the menu. Each entry tells a client (or an LLM driving a client) what the agent can do. Skills are intentionally lightweight in A2A: an `id`, a human-friendly `name` and `description`, tags for routing/categorization, and example invocations.

Notice what's **not** here: full JSON schemas for inputs and outputs. The A2A spec leaves the shape of a skill's input to the **message contents** at call time — A2A messages carry parts (text, files, structured data) rather than enforcing per-skill argument schemas the way an RPC API would. This is a deliberate design choice: agents handle free-form conversational input, not tightly-typed function calls. (If you've worked through the MCP series in this repo, contrast this with MCP tools, which **do** declare JSON Schemas for their inputs.)

In [5]:
for skill in card.skills:
    print(f"• {skill.id}: {skill.name}")
    print(f"    {skill.description}")
    print(f"    tags     = {skill.tags}")
    print(f"    examples = {skill.examples}")

• research_topic: Research a topic
    Given a topic name, return a list of facts about it.
    tags     = ['research', 'facts']
    examples = ['octopuses', 'rome']


## 6. Agent Card vs. OpenAPI

FastAPI gives us OpenAPI for free at `/openapi.json`. That document also describes our service in a machine-readable way. So — why does A2A need its own format?

The short answer: **they describe different things at different abstraction levels.**

| | OpenAPI | A2A Agent Card |
|---|---|---|
| **Describes** | HTTP routes, methods, request/response shapes | An agent's identity, capabilities, skills |
| **Granularity** | Per-endpoint | Per-agent |
| **Input model** | Typed per-route arguments | Free-form messages (text, files, data) |
| **Auth model** | `securitySchemes` + `security` (OpenAPI) | `securitySchemes` + `security` (A2A reuses OpenAPI's shape) |
| **Audience** | Code generators, HTTP clients | Other agents (and the LLMs/runtimes driving them) |
| **Discoverable at** | `/openapi.json` | `/.well-known/agent-card.json` |

A2A deliberately borrows OpenAPI's security model verbatim — that's why this row reads identically on both sides. The rest of the card is its own thing.

OpenAPI is great if you're writing a typed HTTP client. A2A is what an LLM-driven coordinator wants when it's asking *"is there an agent here, and what is it good for?"* In practice both can coexist on the same FastAPI app — and they do, right now, on this notebook's researcher.

In [6]:
openapi = httpx.get("http://127.0.0.1:8010/openapi.json").json()

print("OpenAPI describes routes:")
for path, methods in openapi["paths"].items():
    for method in methods:
        print(f"  {method.upper():6} {path}")

print()
print(f"Agent Card describes one agent named {card.name!r}")
print(f"with {len(card.skills)} skill(s): {[s.id for s in card.skills]}")

OpenAPI describes routes:
  GET    /.well-known/agent-card.json

Agent Card describes one agent named 'Researcher'
with 1 skill(s): ['research_topic']


## What you just learned

- The Agent Card is the entry point of A2A — a single JSON document at `/.well-known/agent-card.json` that describes an agent's identity, capabilities, security, and skills.
- The card is typed: a `pydantic` model on both sides keeps the shape stable.
- A2A reuses OpenAPI's `securitySchemes` + `security` modeling — the auth story is OpenAPI, not a new invention.
- A client only needs the agent's base URL to discover everything else.
- Agent Card and OpenAPI describe **different things** at different abstraction levels — and they coexist happily.

## What's missing

We have an Agent Card. We know what the agent **claims** to do. But we haven't actually **called** the agent yet — there's no way for a client to say *"please research octopuses for me"* and get a result back.

In **notebook 03** we introduce the first real A2A protocol exchange: `message/send`. We'll wrap our researcher's logic behind a JSON-RPC 2.0 endpoint, hand-build the request envelope, and watch the agent return a structured `Task` with an `Artifact` containing the answer.

In [7]:
shutdown_all_servers()
print("All servers stopped.")

All servers stopped.
